# 1. Skup koji ukljucuje drzave i njegove redukcije
U ovom notebook-u cemo se baviti samim klasterovanjem fudbalera. Sprovescemo 5 razlicitih algoritama nad svakim od prethodno izdvojenih skupova podataka: ceo skup, skup sa samo 30 najucestalijih drzava, skup bez atributa vezanih za klubove i PCA od 90%.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

DATASET_DIR = Path("../data/processed/clusters")

X_full = pd.read_csv(DATASET_DIR / "X_full.csv")
X_top30 = pd.read_csv(DATASET_DIR / "X_top30.csv")
X_player_only = pd.read_csv(DATASET_DIR / "X_player_only.csv")
X_pca90 = pd.read_csv(DATASET_DIR / "X_pca90.csv")

player_ids = pd.read_csv(DATASET_DIR / "player_ids_full.csv")

print(X_full.shape)
print(X_top30.shape)
print(X_player_only.shape)
print(X_pca90.shape)

(44905, 352)
(44905, 183)
(44905, 177)
(44905, 82)


In [2]:
# Biblioteke i funkcija za evaluaciju klasterovanja

from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)

def evaluate_clustering(X, labels):
    labels = np.asarray(labels)

    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = np.sum(labels == -1)

    if n_clusters < 2:
        return {
            "n_clusters": n_clusters,
            "n_noise": n_noise,
            "silhouette": np.nan,
            "davies_bouldin": np.nan,
            "calinski_harabasz": np.nan
        }

    mask = labels != -1

    return {
        "n_clusters": n_clusters,
        "n_noise": n_noise,
        "silhouette": silhouette_score(X[mask], labels[mask]),
        "davies_bouldin": davies_bouldin_score(X[mask], labels[mask]),
        "calinski_harabasz": calinski_harabasz_score(X[mask], labels[mask])
    }

In [3]:
# Izdvajanje rezultata u poseban direktorijum i funkcija za njihovo cuvanje

RESULTS_DIR = DATASET_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

def save_result(dataset_name, algorithm_name, labels, metrics):
    cluster_df = pd.DataFrame({
        "player_id": player_ids["player_id"],
        "cluster": labels
    })

    cluster_path = RESULTS_DIR / f"{dataset_name}_{algorithm_name}_clusters.csv"
    cluster_df.to_csv(cluster_path, index=False)

    metrics_df = pd.DataFrame([{
        "dataset": dataset_name,
        "algorithm": algorithm_name,
        **metrics
    }])

    metrics_path = RESULTS_DIR / f"{dataset_name}_{algorithm_name}_metrics.csv"
    metrics_df.to_csv(metrics_path, index=False)

    return metrics_df

In [4]:
# Funkcija za pokretanje bilo kog algoritma klasterovanja nad skupom podataka

def run_algorithm(X, dataset_name, algorithm_name, model):
    print(f"Running {algorithm_name} on {dataset_name}...")
    start = time.time()

    labels = model.fit_predict(X)

    elapsed = time.time() - start
    metrics = evaluate_clustering(X, labels)
    metrics["time_seconds"] = elapsed

    result = save_result(dataset_name, algorithm_name, labels, metrics)

    print(f"Finished {algorithm_name} on {dataset_name} in {elapsed:.2f}s")
    display(result)

    return labels, result

In [5]:
import time

from pathlib import Path

from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN, SpectralClustering
from sklearn.mixture import GaussianMixture

## 1.1 Ceo skup podataka
Najpre cemo pokrenuti svih 5 algoritama nad celim skupom podataka

In [6]:
X_full = pd.read_csv(DATASET_DIR / "X_full.csv").values
X_full.shape

(44905, 352)

In [13]:
full_results = []

### 1.1.1 KMeans

In [14]:
for k in [3,4,5,6,7,10]:
    labels, res = run_algorithm(
        X_full,
        "full",
        "kmeans",
        KMeans(n_clusters=k, random_state=42, n_init=10)
    )
    full_results.append(res)

Running kmeans on full...
Finished kmeans on full in 3.27s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,3,0,0.092998,3.557815,1370.431397,3.271716


Running kmeans on full...
Finished kmeans on full in 3.83s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,4,0,0.022933,4.688854,1103.518366,3.832513


Running kmeans on full...
Finished kmeans on full in 4.53s


...

,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,6,0,-0.009303,4.411299,839.421962,5.443851


Running kmeans on full...
Finished kmeans on full in 4.60s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,7,0,0.010904,3.978885,715.981439,4.604376


Running kmeans on full...
Finished kmeans on full in 5.48s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,10,0,0.00288,4.064461,586.717523,5.48084


In [16]:
labels, res = run_algorithm(
        X_full,
        "full",
        "kmeans",
        KMeans(n_clusters=2, random_state=42, n_init=10)
    )
full_results.append(res)

Running kmeans on full...
Finished kmeans on full in 2.40s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,kmeans,2,0,0.285904,2.482611,1794.611077,2.398051


In [17]:
labels = KMeans(
    n_clusters=2,
    random_state=42,
    n_init=10
).fit_predict(X_full)

pd.Series(labels).value_counts()

1    41968
0     2937
Name: count, dtype: int64

Mozemo videti da sto je vise klastera to su metrike losije. Narocito veliki gubitak se desava na skoku izmedju 2 i 3 i izmedju 3 i 4 klastera. Za 2 i 3 je logicno jer 2 klastera cesto oznacavaju samo jednu trivijalnu podelu kojom se ekstremni primerci izdvajaju od ostalih, sto mozemo i videti iz cinjenice da se u prvom klasteru nalazi svega oko 3000 igraca, dok je u drugom skoro 42000. Sto se tice podele na 3 klastera, ona ima smisla jer se i igraci sami prirodno dele na golmane+defanzivce, vezne igrace i napadace.

In [22]:
full_metrics = pd.concat(full_results, ignore_index=True)
full_metrics = full_metrics.sort_values("n_clusters")

In [23]:
full_metrics

,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
6,full,kmeans,2,0,0.285904,2.482611,1794.611077,2.398051
0,full,kmeans,3,0,0.092998,3.557815,1370.431397,3.271716
1,full,kmeans,4,0,0.022933,4.688854,1103.518366,3.832513
2,full,kmeans,5,0,0.008798,4.736076,914.308726,4.533548
3,full,kmeans,6,0,-0.009303,4.411299,839.421962,5.443851
4,full,kmeans,7,0,0.010904,3.978885,715.981439,4.604376
5,full,kmeans,10,0,0.002880,4.064461,586.717523,5.480840


### 1.1.2 GMM

In [24]:
def run_gmm_grid(X, dataset_name, component_values):
    results = []
    
    for n in component_values:
        print(f"Running GMM on {dataset_name}, n_components={n}...")
        start = time.time()
        
        model = GaussianMixture(
            n_components=n,
            random_state=42,
            covariance_type="full"
        )
        
        labels = model.fit_predict(X)
        elapsed = time.time() - start
        
        metrics = evaluate_clustering(X, labels)
        metrics["time_seconds"] = elapsed
        metrics["dataset"] = dataset_name
        metrics["algorithm"] = "gmm"
        metrics["param"] = f"n_components={n}"
        
        save_result(
            dataset_name,
            f"gmm_{n}",
            labels,
            metrics
        )
        
        results.append(metrics)
        print(f"Finished GMM n_components={n} in {elapsed:.2f}s")
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        by=["silhouette", "davies_bouldin"],
        ascending=[False, True]
    )
    
    results_df.to_csv(
        RESULTS_DIR / f"{dataset_name}_gmm_grid_metrics.csv",
        index=False
    )
    
    return results_df

In [26]:
gmm_values = [2, 3, 4, 5, 6, 7, 10]

full_gmm_metrics = run_gmm_grid(
    X_full,
    "full",
    gmm_values
)

full_gmm_metrics

Running GMM on full, n_components=2...
Finished GMM n_components=2 in 17.92s
Running GMM on full, n_components=3...
Finished GMM n_components=3 in 79.02s
Running GMM on full, n_components=4...
Finished GMM n_components=4 in 197.96s
Running GMM on full, n_components=5...
Finished GMM n_components=5 in 86.70s
Running GMM on full, n_components=6...
Finished GMM n_components=6 in 147.29s
Running GMM on full, n_components=7...
Finished GMM n_components=7 in 172.20s
Running GMM on full, n_components=10...
Finished GMM n_components=10 in 245.06s


,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds,dataset,algorithm,param
0,2,0,0.146481,8.980445,273.376279,17.920218,full,gmm,n_components=2
1,3,0,-0.007107,10.182657,211.752884,79.020835,full,gmm,n_components=3
2,4,0,-0.007938,8.213187,260.056970,197.959175,full,gmm,n_components=4
5,7,0,-0.027530,10.408528,182.754051,172.202639,full,gmm,n_components=7
6,10,0,-0.047322,9.715321,179.206265,245.062071,full,gmm,n_components=10
4,6,0,-0.049566,9.411682,194.096649,147.289478,full,gmm,n_components=6
3,5,0,-0.057939,10.994819,169.447407,86.704471,full,gmm,n_components=5


### 1.1.3 Hierarchical

In [21]:
def run_hierarchical_grid(X, dataset_name, k_values):
    results = []
    
    for k in k_values:
        print(f"Running Hierarchical on {dataset_name}, k={k}...")
        start = time.time()
        
        model = AgglomerativeClustering(
            n_clusters=k,
            linkage="ward"
        )
        
        labels = model.fit_predict(X)
        elapsed = time.time() - start
        
        metrics = evaluate_clustering(X, labels)
        metrics["time_seconds"] = elapsed
        metrics["dataset"] = dataset_name
        metrics["algorithm"] = "hierarchical"
        metrics["param"] = f"n_clusters={k}"
        
        save_result(
            dataset_name,
            f"hierarchical_{k}",
            labels,
            metrics
        )
        
        results.append(metrics)
        print(f"Finished Hierarchical k={k} in {elapsed:.2f}s")
    
    results_df = pd.DataFrame(results)
    results_df = results_df.sort_values(
        by=["silhouette", "davies_bouldin"],
        ascending=[False, True]
    )
    
    results_df.to_csv(
        RESULTS_DIR / f"{dataset_name}_hierarchical_grid_metrics.csv",
        index=False
    )
    
    return results_df

In [ ]:
# Na ovaj nacin bismo pokrenuli hijerarhijsko klasterovanje nad celim skupom podataka, ali nazalost memorijska ogranicenja
# masine ne dozvoljavaju da se ovaj algoritam pokrece nad toliko velikim skupom podataka. Iz tog razloga, ispod cemo pokrenuti
# hijerarhijsko klasterovanje nad probranim skupom, nastalim od celokupnog skupa podataka. U njemu ce biti 10000 instanci.

hierarchical_values = [2, 3, 4, 5, 6, 7]

full_hierarchical_metrics = run_hierarchical_grid(
    X_full,
    "full",
    hierarchical_values
)

full_hierarchical_metrics

Running Hierarchical on full, k=2...


In [17]:
df = pd.read_csv("../data/processed/player_features_full.csv")

df["position"]

0            Attack
1        Goalkeeper
2            Attack
3          Defender
4        Goalkeeper
            ...    
44900      Defender
44901        Attack
44902       Missing
44903       Missing
44904       Missing
Name: position, Length: 44905, dtype: str

In [18]:
from sklearn.model_selection import train_test_split

sample_size = 10000

sample_df, _ = train_test_split(
    df,
    train_size=sample_size,
    random_state=42,
    stratify=df["position"]
)

sample_indices = sample_df.index

In [19]:
X_full_sample = X_full[sample_indices]
player_ids_sample = player_ids.iloc[sample_indices].reset_index(drop=True)

print(X_full_sample.shape)
print(player_ids_sample.shape)

(10000, 352)
(10000, 1)


In [24]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

full_hierarchical_results = []

for k in hierarchical_values:
    labels, res = run_algorithm(
        X_full_sample,
        "full_sample10000",
        f"hierarchical_{k}",
        AgglomerativeClustering(n_clusters=k, linkage="ward")
    )
    full_hierarchical_results.append(res)

player_ids = player_ids_original.copy()

full_hierarchical_metrics = pd.concat(
    full_hierarchical_results,
    ignore_index=True
)

full_hierarchical_metrics

Running hierarchical_2 on full_sample10000...
Finished hierarchical_2 on full_sample10000 in 10.23s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,hierarchical_2,2,0,0.183701,3.190865,285.620691,10.231558


Running hierarchical_3 on full_sample10000...
Finished hierarchical_3 on full_sample10000 in 10.36s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,hierarchical_3,3,0,0.165708,2.521524,220.034967,10.356838


Running hierarchical_4 on full_sample10000...
Finished hierarchical_4 on full_sample10000 in 10.37s


...

Running hierarchical_7 on full_sample10000...
Finished hierarchical_7 on full_sample10000 in 10.19s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,hierarchical_7,7,0,0.164418,1.512764,171.369296,10.189429


Running hierarchical_10 on full_sample10000...
Finished hierarchical_10 on full_sample10000 in 10.40s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,hierarchical_10,10,0,0.166201,0.987707,161.967956,10.400601


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,hierarchical_2,2,0,0.183701,3.190865,285.620691,10.231558
1,full_sample10000,hierarchical_3,3,0,0.165708,2.521524,220.034967,10.356838
2,full_sample10000,hierarchical_4,4,0,0.160574,1.945655,193.799498,10.371940
3,full_sample10000,hierarchical_5,5,0,0.161685,1.633708,181.581006,9.931626
4,full_sample10000,hierarchical_6,6,0,0.163306,1.659640,175.045732,10.034892
5,full_sample10000,hierarchical_7,7,0,0.164418,1.512764,171.369296,10.189429
6,full_sample10000,hierarchical_10,10,0,0.166201,0.987707,161.967956,10.400601


### 1.1.4 Spectral

In [25]:
# Kao i za hijerarhijsko klasterovanje, iz memorijskih razloga sam primoran da radim na smanjenom skupu od 10000 instanci.

from sklearn.cluster import SpectralClustering

player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

spectral_values = [2, 3, 4, 5, 6, 7, 10]

full_spectral_results = []

for k in spectral_values:
    labels, res = run_algorithm(
        X_full_sample,
        "full_sample10000",
        f"spectral_{k}",
        SpectralClustering(
            n_clusters=k,
            random_state=42,
            affinity="nearest_neighbors",
            n_neighbors=10,
            assign_labels="kmeans"
        )
    )
    full_spectral_results.append(res)

player_ids = player_ids_original.copy()

full_spectral_metrics = pd.concat(
    full_spectral_results,
    ignore_index=True
)

full_spectral_metrics

Running spectral_2 on full_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_2 on full_sample10000 in 4.04s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,spectral_2,2,0,0.353426,0.967639,23.880402,4.036591


Running spectral_3 on full_sample10000...


...

Running spectral_10 on full_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_10 on full_sample10000 in 6.64s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,spectral_10,10,0,0.204345,1.035316,32.766579,6.638618


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full_sample10000,spectral_2,2,0,0.353426,0.967639,23.880402,4.036591
1,full_sample10000,spectral_3,3,0,0.354449,0.900331,29.002951,5.006272
2,full_sample10000,spectral_4,4,0,0.195234,1.084282,30.043607,8.274833
3,full_sample10000,spectral_5,5,0,0.196869,1.052619,32.370604,7.037839
4,full_sample10000,spectral_6,6,0,0.198155,1.028901,31.286940,5.964190
5,full_sample10000,spectral_7,7,0,0.199791,1.013679,32.715946,8.728800
6,full_sample10000,spectral_10,10,0,0.204345,1.035316,32.766579,6.638618


### 1.1.5 DBSCAN

In [26]:
full_dbscan_results = []

eps_values = [2.0, 2.5, 3.0, 3.5, 4.0]
min_samples_values = [5, 10, 20]

for eps in eps_values:
    for min_samples in min_samples_values:

        labels, res = run_algorithm(
            X_full,
            "full",
            f"dbscan_eps{eps}_min{min_samples}",
            DBSCAN(
                eps=eps,
                min_samples=min_samples,
                n_jobs=-1
            )
        )

        res["param"] = (
            f"eps={eps}, "
            f"min_samples={min_samples}"
        )

        full_dbscan_results.append(res)

full_dbscan_metrics = pd.concat(
    full_dbscan_results,
    ignore_index=True
)

full_dbscan_metrics = full_dbscan_metrics.sort_values(
    by=["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

full_dbscan_metrics

Running dbscan_eps2.0_min5 on full...
Finished dbscan_eps2.0_min5 on full in 11.38s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,dbscan_eps2.0_min5,242,42280,0.505377,0.676033,391.0826,11.376382


Running dbscan_eps2.0_min10 on full...
Finished dbscan_eps2.0_min10 on full in 8.08s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,dbscan_eps2.0_min10,43,44019,0.615637,0.516734,630.171124,8.080486


Running dbscan_eps2.0_min20 on full...
Finished dbscan_eps2.0_min20 on full in 8.91s


...

Running dbscan_eps4.0_min10 on full...
Finished dbscan_eps4.0_min10 on full in 11.35s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,dbscan_eps4.0_min10,310,33522,0.35547,1.072358,295.141625,11.346353


Running dbscan_eps4.0_min20 on full...
Finished dbscan_eps4.0_min20 on full in 11.89s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,full,dbscan_eps4.0_min20,129,38351,0.422953,0.955844,412.823657,11.89427


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds,param
2,full,dbscan_eps2.0_min20,4,44795,0.836545,0.227931,1545.597210,8.907377,"eps=2.0, min_samples=20"
5,full,dbscan_eps2.5_min20,21,44140,0.621737,0.557091,680.393036,10.447626,"eps=2.5, min_samples=20"
1,full,dbscan_eps2.0_min10,43,44019,0.615637,0.516734,630.171124,8.080486,"eps=2.0, min_samples=10"
8,full,dbscan_eps3.0_min20,46,43219,0.527369,0.753301,541.788972,10.080527,"eps=3.0, min_samples=20"
4,full,dbscan_eps2.5_min10,116,42473,0.512243,0.730510,473.951979,10.143268,"eps=2.5, min_samples=10"
0,full,dbscan_eps2.0_min5,242,42280,0.505377,0.676033,391.082600,11.376382,"eps=2.0, min_samples=5"
11,full,dbscan_eps3.5_min20,78,41472,0.467873,0.848320,476.067075,10.805699,"eps=3.5, min_samples=20"
3,full,dbscan_eps2.5_min5,445,39554,0.434566,0.825888,301.056998,9.171053,"eps=2.5, min_samples=5"
7,full,dbscan_eps3.0_min10,219,40139,0.423237,0.893757,366.760748,9.923684,"eps=3.0, min_samples=10"
14,full,dbscan_eps4.0_min20,129,38351,0.422953,0.955844,412.823657,11.894270,"eps=4.0, min_samples=20"


Na ovolikom skupu, sa 364 atributa, logicno je da DBSCAN ne moze da pronadje lepe klastere, pogotovo sto je susedstvo u prostoru sa 364 dimenzija veoma komplikovano odrediti. Zbog toga mozemo videti, da iako je u nekim slucajevima silhouette skor veoma visok (cak preko 0.8), DBSCAN je u te klastere uspeo da ubaci cak manje od 1% svih igraca, dok su ostali oznaceni kao sum. Ovi rezultati ce se popraviti kako budemo radili nad redukovanim skupovima podataka.

## 1.2. Skup podataka sa 30 najzastupljenijih drzava

In [27]:
X_top30.shape

(44905, 183)

### 1.2.1 KMeans

In [28]:
top30_kmeans_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:
    labels, res = run_algorithm(
        X_top30,
        "top30",
        f"kmeans_{k}",
        KMeans(n_clusters=k, random_state=42, n_init=10)
    )
    top30_kmeans_results.append(res)

top30_kmeans_metrics = pd.concat(
    top30_kmeans_results,
    ignore_index=True
)

top30_kmeans_metrics

Running kmeans_2 on top30...
Finished kmeans_2 on top30 in 3.51s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,kmeans_2,2,0,0.365382,2.153653,3527.538858,3.514682


Running kmeans_3 on top30...
Finished kmeans_3 on top30 in 2.80s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,kmeans_3,3,0,0.126104,2.929543,2746.432205,2.801903


Running kmeans_4 on top30...
Finished kmeans_4 on top30 in 2.75s


...

Running kmeans_7 on top30...
Finished kmeans_7 on top30 in 3.37s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,kmeans_7,7,0,0.048294,3.366813,1571.38015,3.369373


Running kmeans_10 on top30...
Finished kmeans_10 on top30 in 3.35s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,kmeans_10,10,0,0.048174,3.063741,1275.827659,3.352907


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,kmeans_2,2,0,0.365382,2.153653,3527.538858,3.514682
1,top30,kmeans_3,3,0,0.126104,2.929543,2746.432205,2.801903
2,top30,kmeans_4,4,0,0.018953,3.003706,2190.805234,2.752932
3,top30,kmeans_5,5,0,0.027068,3.657179,1934.608084,3.115752
4,top30,kmeans_6,6,0,0.034176,3.630004,1710.492562,3.120263
5,top30,kmeans_7,7,0,0.048294,3.366813,1571.380150,3.369373
6,top30,kmeans_10,10,0,0.048174,3.063741,1275.827659,3.352907


### 1.2.2 GMM

In [29]:
top30_gmm_results = []

for n in [2, 3, 4, 5, 6, 7, 10]:
    labels, res = run_algorithm(
        X_top30,
        "top30",
        f"gmm_{n}",
        GaussianMixture(
            n_components=n,
            random_state=42,
            covariance_type="full"
        )
    )
    top30_gmm_results.append(res)

top30_gmm_metrics = pd.concat(
    top30_gmm_results,
    ignore_index=True
)

top30_gmm_metrics

Running gmm_2 on top30...
Finished gmm_2 on top30 in 14.85s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,gmm_2,2,0,0.01427,9.310059,434.153007,14.851001


Running gmm_3 on top30...
Finished gmm_3 on top30 in 16.22s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,gmm_3,3,0,0.003263,7.664281,545.785561,16.223021


Running gmm_4 on top30...
Finished gmm_4 on top30 in 25.59s


...

Running gmm_7 on top30...
Finished gmm_7 on top30 in 64.56s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,gmm_7,7,0,-0.02248,5.464753,551.390711,64.557343


Running gmm_10 on top30...
Finished gmm_10 on top30 in 68.30s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,gmm_10,10,0,-0.012543,4.862574,559.365981,68.302126


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,gmm_2,2,0,0.014270,9.310059,434.153007,14.851001
1,top30,gmm_3,3,0,0.003263,7.664281,545.785561,16.223021
2,top30,gmm_4,4,0,-0.008377,5.707053,949.028729,25.590941
3,top30,gmm_5,5,0,-0.030277,6.683362,561.498352,23.259484
4,top30,gmm_6,6,0,-0.022530,5.876076,501.643416,66.478003
5,top30,gmm_7,7,0,-0.022480,5.464753,551.390711,64.557343
6,top30,gmm_10,10,0,-0.012543,4.862574,559.365981,68.302126


### 1.2.3 Hierarchical

In [31]:
# Kao i za ceo skup, zbog broja instanci, memorijska ogranicenja ne mogu biti zadovoljena. Iz tog razloga cemo i ovde koristiti
# skup smanjen na 10000 instanci prilikom hijerarhijskog i spektralnog klasterovanja.

df_meta = pd.read_csv("../data/processed/player_features_full.csv")

from sklearn.model_selection import train_test_split

sample_size = 10000

sample_meta, _ = train_test_split(
    df_meta,
    train_size=sample_size,
    random_state=42,
    stratify=df_meta["position"]
)

sample_indices = sample_meta.index.to_numpy()

X_top30_np = pd.read_csv(DATASET_DIR / "X_top30.csv").values

X_top30_sample = X_top30_np[sample_indices]
player_ids_sample = player_ids.iloc[sample_indices].reset_index(drop=True)

print(X_top30_sample.shape)
print(player_ids_sample.shape)

(10000, 183)
(10000, 1)


In [33]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

top30_hierarchical_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:
    labels, res = run_algorithm(
        X_top30_sample,
        "top30_sample10000",
        f"hierarchical_{k}",
        AgglomerativeClustering(
            n_clusters=k,
            linkage="ward"
        )
    )
    top30_hierarchical_results.append(res)

player_ids = player_ids_original.copy()

top30_hierarchical_metrics = pd.concat(
    top30_hierarchical_results,
    ignore_index=True
)

top30_hierarchical_metrics

Running hierarchical_2 on top30_sample10000...
Finished hierarchical_2 on top30_sample10000 in 5.67s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,hierarchical_2,2,0,0.229882,2.902698,611.767296,5.666476


Running hierarchical_3 on top30_sample10000...
Finished hierarchical_3 on top30_sample10000 in 5.82s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,hierarchical_3,3,0,0.182663,2.643089,510.35771,5.816978


Running hierarchical_4 on top30_sample10000...
Finished hierarchical_4 on top30_sample10000 in 6.06s


...

Running hierarchical_7 on top30_sample10000...
Finished hierarchical_7 on top30_sample10000 in 7.32s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,hierarchical_7,7,0,0.053072,3.220681,277.658162,7.323346


Running hierarchical_10 on top30_sample10000...
Finished hierarchical_10 on top30_sample10000 in 6.30s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,hierarchical_10,10,0,-0.00679,2.91604,227.277245,6.297654


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,hierarchical_2,2,0,0.229882,2.902698,611.767296,5.666476
1,top30_sample10000,hierarchical_3,3,0,0.182663,2.643089,510.357710,5.816978
2,top30_sample10000,hierarchical_4,4,0,0.183343,2.351608,399.486004,6.056973
3,top30_sample10000,hierarchical_5,5,0,0.187024,2.267322,344.789020,6.118239
4,top30_sample10000,hierarchical_6,6,0,0.048835,3.512762,306.918666,6.101719
5,top30_sample10000,hierarchical_7,7,0,0.053072,3.220681,277.658162,7.323346
6,top30_sample10000,hierarchical_10,10,0,-0.006790,2.916040,227.277245,6.297654


### 1.2.4 Spectral

In [34]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

top30_spectral_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:
    labels, res = run_algorithm(
        X_top30_sample,
        "top30_sample10000",
        f"spectral_{k}",
        SpectralClustering(
            n_clusters=k,
            random_state=42,
            affinity="nearest_neighbors",
            n_neighbors=10,
            assign_labels="kmeans"
        )
    )
    top30_spectral_results.append(res)

player_ids = player_ids_original.copy()

top30_spectral_metrics = pd.concat(
    top30_spectral_results,
    ignore_index=True
)

top30_spectral_metrics

Running spectral_2 on top30_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_2 on top30_sample10000 in 4.55s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,spectral_2,2,0,0.249712,0.720178,21.501418,4.550349


Running spectral_3 on top30_sample10000...


...

Running spectral_10 on top30_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_10 on top30_sample10000 in 4.88s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,spectral_10,10,0,-0.088562,1.625134,59.765083,4.878345


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30_sample10000,spectral_2,2,0,0.249712,0.720178,21.501418,4.550349
1,top30_sample10000,spectral_3,3,0,0.252482,2.270382,66.293815,4.770426
2,top30_sample10000,spectral_4,4,0,0.025210,2.151671,66.137032,4.807539
3,top30_sample10000,spectral_5,5,0,0.026779,2.114614,62.575869,4.820892
4,top30_sample10000,spectral_6,6,0,0.034453,2.026126,73.240838,4.745060
5,top30_sample10000,spectral_7,7,0,0.037280,1.952709,74.104382,4.920335
6,top30_sample10000,spectral_10,10,0,-0.088562,1.625134,59.765083,4.878345


### 1.2.5 DBSCAN

In [36]:
player_ids = pd.read_csv(DATASET_DIR / "player_ids_full.csv")

top30_dbscan_results = []

eps_values = [2.0, 2.5, 3.0, 3.5, 4.0]
min_samples_values = [5, 10, 20]

for eps in eps_values:
    for min_samples in min_samples_values:
        labels, res = run_algorithm(
            X_top30,
            "top30",
            f"dbscan_eps{eps}_min{min_samples}",
            DBSCAN(
                eps=eps,
                min_samples=min_samples,
                n_jobs=-1
            )
        )

        res["param"] = f"eps={eps}, min_samples={min_samples}"
        top30_dbscan_results.append(res)

top30_dbscan_metrics = pd.concat(
    top30_dbscan_results,
    ignore_index=True
)

top30_dbscan_metrics = top30_dbscan_metrics.sort_values(
    by=["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

top30_dbscan_metrics

Running dbscan_eps2.0_min5 on top30...
Finished dbscan_eps2.0_min5 on top30 in 7.77s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,dbscan_eps2.0_min5,252,42223,0.508968,0.666925,371.004383,7.765723


Running dbscan_eps2.0_min10 on top30...
Finished dbscan_eps2.0_min10 on top30 in 7.73s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,dbscan_eps2.0_min10,43,44018,0.615471,0.518106,621.368946,7.733962


Running dbscan_eps2.0_min20 on top30...
Finished dbscan_eps2.0_min20 on top30 in 4.80s


...

Running dbscan_eps4.0_min10 on top30...
Finished dbscan_eps4.0_min10 on top30 in 5.46s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,dbscan_eps4.0_min10,364,32362,0.344965,1.078759,261.411917,5.458238


Running dbscan_eps4.0_min20 on top30...
Finished dbscan_eps4.0_min20 on top30 in 5.37s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,top30,dbscan_eps4.0_min20,135,38130,0.419975,0.966258,389.875554,5.374076


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds,param
2,top30,dbscan_eps2.0_min20,4,44795,0.836545,0.227931,1545.597210,4.802134,"eps=2.0, min_samples=20"
5,top30,dbscan_eps2.5_min20,21,44140,0.621737,0.557091,680.393036,4.656730,"eps=2.5, min_samples=20"
1,top30,dbscan_eps2.0_min10,43,44018,0.615471,0.518106,621.368946,7.733962,"eps=2.0, min_samples=10"
8,top30,dbscan_eps3.0_min20,46,43219,0.527369,0.753301,541.788972,5.227837,"eps=3.0, min_samples=20"
4,top30,dbscan_eps2.5_min10,119,42435,0.512621,0.727439,461.084578,4.483177,"eps=2.5, min_samples=10"
0,top30,dbscan_eps2.0_min5,252,42223,0.508968,0.666925,371.004383,7.765723,"eps=2.0, min_samples=5"
11,top30,dbscan_eps3.5_min20,80,41405,0.464427,0.860642,455.652563,5.300107,"eps=3.5, min_samples=20"
3,top30,dbscan_eps2.5_min5,472,39338,0.437972,0.817956,283.123300,4.171511,"eps=2.5, min_samples=5"
7,top30,dbscan_eps3.0_min10,230,39978,0.423896,0.892377,346.147135,5.005268,"eps=3.0, min_samples=10"
14,top30,dbscan_eps4.0_min20,135,38130,0.419975,0.966258,389.875554,5.374076,"eps=4.0, min_samples=20"


## 1.3. Player only skup podataka (skup bez atributa vezanih za klubove)

In [37]:
X_player_only = pd.read_csv(
    DATASET_DIR / "X_player_only.csv"
).values

X_player_only.shape

(44905, 177)

### 1.3.1 KMeans

In [38]:
player_only_kmeans_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_player_only,
        "player_only",
        f"kmeans_{k}",
        KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
    )

    player_only_kmeans_results.append(res)

player_only_kmeans_metrics = pd.concat(
    player_only_kmeans_results,
    ignore_index=True
)

player_only_kmeans_metrics

Running kmeans_2 on player_only...
Finished kmeans_2 on player_only in 2.15s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,kmeans_2,2,0,0.373489,2.13885,3599.400806,2.150467


Running kmeans_3 on player_only...
Finished kmeans_3 on player_only in 2.59s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,kmeans_3,3,0,0.1304,2.895698,2816.19037,2.586435


Running kmeans_4 on player_only...
Finished kmeans_4 on player_only in 2.66s


...

Running kmeans_7 on player_only...
Finished kmeans_7 on player_only in 3.06s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,kmeans_7,7,0,0.041595,3.1468,1583.204593,3.05851


Running kmeans_10 on player_only...
Finished kmeans_10 on player_only in 3.42s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,kmeans_10,10,0,0.054759,3.02505,1289.266487,3.419992


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,kmeans_2,2,0,0.373489,2.138850,3599.400806,2.150467
1,player_only,kmeans_3,3,0,0.130400,2.895698,2816.190370,2.586435
2,player_only,kmeans_4,4,0,0.027212,3.707816,2291.310014,2.662862
3,player_only,kmeans_5,5,0,0.023409,3.688615,1911.730224,2.561258
4,player_only,kmeans_6,6,0,0.034239,3.567222,1729.877091,3.122943
5,player_only,kmeans_7,7,0,0.041595,3.146800,1583.204593,3.058510
6,player_only,kmeans_10,10,0,0.054759,3.025050,1289.266487,3.419992


### 1.3.2 GMM

In [39]:
player_only_gmm_results = []

for n in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_player_only,
        "player_only",
        f"gmm_{n}",
        GaussianMixture(
            n_components=n,
            random_state=42,
            covariance_type="full"
        )
    )

    player_only_gmm_results.append(res)

player_only_gmm_metrics = pd.concat(
    player_only_gmm_results,
    ignore_index=True
)

player_only_gmm_metrics

Running gmm_2 on player_only...
Finished gmm_2 on player_only in 17.10s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,gmm_2,2,0,0.01398,9.528311,410.874534,17.101738


Running gmm_3 on player_only...
Finished gmm_3 on player_only in 8.37s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,gmm_3,3,0,0.031249,4.757001,624.038364,8.370882


Running gmm_4 on player_only...
Finished gmm_4 on player_only in 14.03s


...

Running gmm_7 on player_only...
Finished gmm_7 on player_only in 52.81s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,gmm_7,7,0,-0.007209,5.35306,630.156818,52.81127


Running gmm_10 on player_only...
Finished gmm_10 on player_only in 90.12s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,gmm_10,10,0,-0.039564,5.450597,498.313834,90.121006


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,gmm_2,2,0,0.013980,9.528311,410.874534,17.101738
1,player_only,gmm_3,3,0,0.031249,4.757001,624.038364,8.370882
2,player_only,gmm_4,4,0,-0.003271,4.410313,760.251261,14.033902
3,player_only,gmm_5,5,0,-0.005191,5.234913,644.439620,22.454240
4,player_only,gmm_6,6,0,-0.009538,5.026055,648.338117,31.688997
5,player_only,gmm_7,7,0,-0.007209,5.353060,630.156818,52.811270
6,player_only,gmm_10,10,0,-0.039564,5.450597,498.313834,90.121006


### 1.3.3 Hierarchical
Ponovo vazi ista prica kao i za prethodne skupove. Za spektralno i hijerarhijsko klasterovanje cemo ostaviti 10000 instanci iz skupa. Sa svim instancama cemo pokusati na PCA verziji, kao i kasnije, na verziji podataka koja uopste nece ukljucivati drzave buduci da one prave ogromnu dimenzionalnost.

In [40]:
df_meta = pd.read_csv(
    "../data/processed/player_features_full.csv"
)

sample_size = 10000

sample_meta, _ = train_test_split(
    df_meta,
    train_size=sample_size,
    random_state=42,
    stratify=df_meta["position"]
)

sample_indices = sample_meta.index.to_numpy()

In [41]:
X_player_only_sample = X_player_only[sample_indices]

player_ids_sample = (
    player_ids
    .iloc[sample_indices]
    .reset_index(drop=True)
)

print(X_player_only_sample.shape)

(10000, 177)


In [42]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

player_only_hierarchical_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_player_only_sample,
        "player_only_sample10000",
        f"hierarchical_{k}",
        AgglomerativeClustering(
            n_clusters=k,
            linkage="ward"
        )
    )

    player_only_hierarchical_results.append(res)

player_ids = player_ids_original.copy()

player_only_hierarchical_metrics = pd.concat(
    player_only_hierarchical_results,
    ignore_index=True
)

player_only_hierarchical_metrics

Running hierarchical_2 on player_only_sample10000...
Finished hierarchical_2 on player_only_sample10000 in 6.01s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,hierarchical_2,2,0,0.238181,2.914526,642.500051,6.011415


Running hierarchical_3 on player_only_sample10000...
Finished hierarchical_3 on player_only_sample10000 in 10.39s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,hierarchical_3,3,0,0.224819,2.31257,495.313086,10.38778


Running hierarchical_4 on player_only_sample10000...
Finished hierarchical_4 on player_only_sample10000 in 6.23s


...

Running hierarchical_7 on player_only_sample10000...
Finished hierarchical_7 on player_only_sample10000 in 7.09s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,hierarchical_7,7,0,0.046094,3.351792,287.820075,7.086717


Running hierarchical_10 on player_only_sample10000...
Finished hierarchical_10 on player_only_sample10000 in 5.88s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,hierarchical_10,10,0,0.004141,3.015916,234.721499,5.879434


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,hierarchical_2,2,0,0.238181,2.914526,642.500051,6.011415
1,player_only_sample10000,hierarchical_3,3,0,0.224819,2.312570,495.313086,10.387780
2,player_only_sample10000,hierarchical_4,4,0,0.185712,2.491742,415.880228,6.225861
3,player_only_sample10000,hierarchical_5,5,0,0.189582,2.372342,358.917432,5.894582
4,player_only_sample10000,hierarchical_6,6,0,0.041079,3.650550,318.329596,6.134908
5,player_only_sample10000,hierarchical_7,7,0,0.046094,3.351792,287.820075,7.086717
6,player_only_sample10000,hierarchical_10,10,0,0.004141,3.015916,234.721499,5.879434


### 1.3.4 Spectral

In [43]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

player_only_spectral_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_player_only_sample,
        "player_only_sample10000",
        f"spectral_{k}",
        SpectralClustering(
            n_clusters=k,
            random_state=42,
            affinity="nearest_neighbors",
            n_neighbors=10,
            assign_labels="kmeans"
        )
    )

    player_only_spectral_results.append(res)

player_ids = player_ids_original.copy()

player_only_spectral_metrics = pd.concat(
    player_only_spectral_results,
    ignore_index=True
)

player_only_spectral_metrics

Running spectral_2 on player_only_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_2 on player_only_sample10000 in 3.55s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,spectral_2,2,0,0.252523,0.667824,21.666614,3.554795


Running spectral_3 on player_only_sample10000...


...

Running spectral_10 on player_only_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_10 on player_only_sample10000 in 4.02s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,spectral_10,10,0,-0.096987,1.987582,58.759614,4.01577


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only_sample10000,spectral_2,2,0,0.252523,0.667824,21.666614,3.554795
1,player_only_sample10000,spectral_3,3,0,0.255465,2.240693,67.778469,5.180932
2,player_only_sample10000,spectral_4,4,0,0.028540,2.126811,67.247053,3.951791
3,player_only_sample10000,spectral_5,5,0,0.031406,1.992292,70.028041,4.261911
4,player_only_sample10000,spectral_6,6,0,-0.073826,1.944880,58.457967,4.271836
5,player_only_sample10000,spectral_7,7,0,-0.070256,1.942759,57.770030,4.233995
6,player_only_sample10000,spectral_10,10,0,-0.096987,1.987582,58.759614,4.015770


### 1.3.5 DBSCAN

In [44]:
len(player_ids)

44905

In [45]:
player_only_dbscan_results = []

eps_values = [2.0, 2.5, 3.0, 3.5, 4.0]
min_samples_values = [5, 10, 20]

for eps in eps_values:
    for min_samples in min_samples_values:

        labels, res = run_algorithm(
            X_player_only,
            "player_only",
            f"dbscan_eps{eps}_min{min_samples}",
            DBSCAN(
                eps=eps,
                min_samples=min_samples,
                n_jobs=-1
            )
        )

        res["param"] = (
            f"eps={eps}, min_samples={min_samples}"
        )

        player_only_dbscan_results.append(res)

player_only_dbscan_metrics = pd.concat(
    player_only_dbscan_results,
    ignore_index=True
)

player_only_dbscan_metrics = player_only_dbscan_metrics.sort_values(
    by=["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

player_only_dbscan_metrics

Running dbscan_eps2.0_min5 on player_only...
Finished dbscan_eps2.0_min5 on player_only in 7.37s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,dbscan_eps2.0_min5,599,37494,0.537534,0.633686,477.28948,7.373297


Running dbscan_eps2.0_min10 on player_only...
Finished dbscan_eps2.0_min10 on player_only in 7.11s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,dbscan_eps2.0_min10,204,40754,0.582711,0.590426,742.330623,7.1083


Running dbscan_eps2.0_min20 on player_only...
Finished dbscan_eps2.0_min20 on player_only in 4.55s


...

Running dbscan_eps4.0_min10 on player_only...
Finished dbscan_eps4.0_min10 on player_only in 5.33s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,dbscan_eps4.0_min10,386,27578,0.380019,1.023454,319.259268,5.333565


Running dbscan_eps4.0_min20 on player_only...
Finished dbscan_eps4.0_min20 on player_only in 5.61s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,player_only,dbscan_eps4.0_min20,199,33073,0.417092,0.971336,433.468619,5.605409


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds,param
2,player_only,dbscan_eps2.0_min20,44,43287,0.673738,0.449729,1193.785661,4.550399,"eps=2.0, min_samples=20"
5,player_only,dbscan_eps2.5_min20,72,42255,0.605144,0.583996,870.065769,4.643397,"eps=2.5, min_samples=20"
1,player_only,dbscan_eps2.0_min10,204,40754,0.582711,0.590426,742.330623,7.108300,"eps=2.0, min_samples=10"
0,player_only,dbscan_eps2.0_min5,599,37494,0.537534,0.633686,477.289480,7.373297,"eps=2.0, min_samples=5"
8,player_only,dbscan_eps3.0_min20,112,40628,0.524402,0.724458,628.375894,4.848544,"eps=3.0, min_samples=20"
4,player_only,dbscan_eps2.5_min10,300,38465,0.511665,0.721992,520.507039,4.770240,"eps=2.5, min_samples=10"
11,player_only,dbscan_eps3.5_min20,152,37391,0.463864,0.844335,504.631501,5.412747,"eps=3.5, min_samples=20"
3,player_only,dbscan_eps2.5_min5,759,34480,0.454378,0.771839,330.070856,4.670101,"eps=2.5, min_samples=5"
7,player_only,dbscan_eps3.0_min10,372,35877,0.441641,0.848555,385.912984,4.777793,"eps=3.0, min_samples=10"
14,player_only,dbscan_eps4.0_min20,199,33073,0.417092,0.971336,433.468619,5.605409,"eps=4.0, min_samples=20"


## 1.4. PCA sa 90% varijanse

In [46]:
X_pca90 = pd.read_csv(
    DATASET_DIR / "X_pca90.csv"
).values

X_pca90.shape

(44905, 82)

### 1.4.1 KMeans

In [47]:
pca_kmeans_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_pca90,
        "pca90",
        f"kmeans_{k}",
        KMeans(
            n_clusters=k,
            random_state=42,
            n_init=10
        )
    )

    pca_kmeans_results.append(res)

pca_kmeans_metrics = pd.concat(
    pca_kmeans_results,
    ignore_index=True
)

pca_kmeans_metrics

Running kmeans_2 on pca90...
Finished kmeans_2 on pca90 in 1.76s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,kmeans_2,2,0,0.382131,2.006672,4032.703253,1.758026


Running kmeans_3 on pca90...
Finished kmeans_3 on pca90 in 2.03s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,kmeans_3,3,0,0.139755,2.686728,3171.378254,2.029605


Running kmeans_4 on pca90...
Finished kmeans_4 on pca90 in 1.99s


...

Running kmeans_7 on pca90...
Finished kmeans_7 on pca90 in 2.45s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,kmeans_7,7,0,0.053252,3.242437,1821.375145,2.449894


Running kmeans_10 on pca90...
Finished kmeans_10 on pca90 in 2.46s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,kmeans_10,10,0,0.052896,2.926616,1530.474736,2.459114


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,kmeans_2,2,0,0.382131,2.006672,4032.703253,1.758026
1,pca90,kmeans_3,3,0,0.139755,2.686728,3171.378254,2.029605
2,pca90,kmeans_4,4,0,0.032723,3.468463,2587.722575,1.994845
3,pca90,kmeans_5,5,0,0.040373,3.367296,2249.778585,2.069509
4,pca90,kmeans_6,6,0,0.046888,3.120892,1930.713465,2.266186
5,pca90,kmeans_7,7,0,0.053252,3.242437,1821.375145,2.449894
6,pca90,kmeans_10,10,0,0.052896,2.926616,1530.474736,2.459114


### 1.4.2 GMM

In [48]:
pca_gmm_results = []

for n in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_pca90,
        "pca90",
        f"gmm_{n}",
        GaussianMixture(
            n_components=n,
            random_state=42,
            covariance_type="full"
        )
    )

    pca_gmm_results.append(res)

pca_gmm_metrics = pd.concat(
    pca_gmm_results,
    ignore_index=True
)

pca_gmm_metrics

Running gmm_2 on pca90...
Finished gmm_2 on pca90 in 4.84s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,gmm_2,2,0,-0.004184,8.671118,446.821213,4.843982


Running gmm_3 on pca90...
Finished gmm_3 on pca90 in 6.34s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,gmm_3,3,0,-0.029491,6.648387,559.946325,6.339532


Running gmm_4 on pca90...
Finished gmm_4 on pca90 in 8.57s


...

Running gmm_7 on pca90...
Finished gmm_7 on pca90 in 34.49s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,gmm_7,7,0,-0.03616,5.445931,599.546513,34.487916


Running gmm_10 on pca90...
Finished gmm_10 on pca90 in 50.07s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,gmm_10,10,0,-0.034019,4.830309,604.218797,50.065438


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,gmm_2,2,0,-0.004184,8.671118,446.821213,4.843982
1,pca90,gmm_3,3,0,-0.029491,6.648387,559.946325,6.339532
2,pca90,gmm_4,4,0,-0.046963,6.799567,538.821830,8.568334
3,pca90,gmm_5,5,0,-0.046950,5.846625,721.926294,16.122610
4,pca90,gmm_6,6,0,-0.044767,5.835795,633.729629,15.248228
5,pca90,gmm_7,7,0,-0.036160,5.445931,599.546513,34.487916
6,pca90,gmm_10,10,0,-0.034019,4.830309,604.218797,50.065438


### 1.4.3 Hierarchical

In [49]:
df_meta = pd.read_csv(
    "../data/processed/player_features_full.csv"
)

sample_size = 10000

sample_meta, _ = train_test_split(
    df_meta,
    train_size=sample_size,
    random_state=42,
    stratify=df_meta["position"]
)

sample_indices = sample_meta.index.to_numpy()

X_pca90_sample = X_pca90[sample_indices]

player_ids_sample = (
    player_ids
    .iloc[sample_indices]
    .reset_index(drop=True)
)

print(X_pca90_sample.shape)

(10000, 82)


In [50]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

pca_hierarchical_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_pca90_sample,
        "pca90_sample10000",
        f"hierarchical_{k}",
        AgglomerativeClustering(
            n_clusters=k,
            linkage="ward"
        )
    )

    pca_hierarchical_results.append(res)

player_ids = player_ids_original.copy()

pca_hierarchical_metrics = pd.concat(
    pca_hierarchical_results,
    ignore_index=True
)

pca_hierarchical_metrics

Running hierarchical_2 on pca90_sample10000...
Finished hierarchical_2 on pca90_sample10000 in 3.54s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,hierarchical_2,2,0,0.2613,2.632552,718.691935,3.544441


Running hierarchical_3 on pca90_sample10000...
Finished hierarchical_3 on pca90_sample10000 in 3.58s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,hierarchical_3,3,0,0.199449,2.596246,549.347317,3.576464


Running hierarchical_4 on pca90_sample10000...
Finished hierarchical_4 on pca90_sample10000 in 5.25s


...

Running hierarchical_7 on pca90_sample10000...
Finished hierarchical_7 on pca90_sample10000 in 3.54s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,hierarchical_7,7,0,0.048958,3.175513,319.972448,3.54303


Running hierarchical_10 on pca90_sample10000...
Finished hierarchical_10 on pca90_sample10000 in 3.54s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,hierarchical_10,10,0,0.012259,2.759291,264.091014,3.544443


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,hierarchical_2,2,0,0.261300,2.632552,718.691935,3.544441
1,pca90_sample10000,hierarchical_3,3,0,0.199449,2.596246,549.347317,3.576464
2,pca90_sample10000,hierarchical_4,4,0,0.201020,2.307812,458.266439,5.248471
3,pca90_sample10000,hierarchical_5,5,0,0.205269,2.207691,396.823564,3.991504
4,pca90_sample10000,hierarchical_6,6,0,0.043594,3.458402,352.889737,3.424023
5,pca90_sample10000,hierarchical_7,7,0,0.048958,3.175513,319.972448,3.543030
6,pca90_sample10000,hierarchical_10,10,0,0.012259,2.759291,264.091014,3.544443


### 1.4.4 Spectral

In [51]:
player_ids_original = player_ids.copy()
player_ids = player_ids_sample.copy()

pca_spectral_results = []

for k in [2, 3, 4, 5, 6, 7, 10]:

    labels, res = run_algorithm(
        X_pca90_sample,
        "pca90_sample10000",
        f"spectral_{k}",
        SpectralClustering(
            n_clusters=k,
            random_state=42,
            affinity="nearest_neighbors",
            n_neighbors=10,
            assign_labels="kmeans"
        )
    )

    pca_spectral_results.append(res)

player_ids = player_ids_original.copy()

pca_spectral_metrics = pd.concat(
    pca_spectral_results,
    ignore_index=True
)

pca_spectral_metrics

Running spectral_2 on pca90_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_2 on pca90_sample10000 in 5.75s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,spectral_2,2,0,0.275931,0.63541,23.33676,5.753997


Running spectral_3 on pca90_sample10000...


...

Running spectral_10 on pca90_sample10000...


/home/janic/projects/ip2-football-players/venv-ip2/lib/python3.12/site-packages/sklearn/manifold/_spectral_embedding.py:325: UserWarning: Graph is not fully connected, spectral embedding may not work as expected.
  warnings.warn(


Finished spectral_10 on pca90_sample10000 in 6.92s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,spectral_10,10,0,-0.070962,2.06564,88.984176,6.920037


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90_sample10000,spectral_2,2,0,0.275931,0.635410,23.336760,5.753997
1,pca90_sample10000,spectral_3,3,0,0.040302,1.337199,47.907474,5.938075
2,pca90_sample10000,spectral_4,4,0,-0.100034,1.405247,38.027932,6.044088
3,pca90_sample10000,spectral_5,5,0,-0.096174,1.517413,43.046678,6.184342
4,pca90_sample10000,spectral_6,6,0,-0.092260,1.577215,45.895198,7.632823
5,pca90_sample10000,spectral_7,7,0,-0.086122,1.728980,65.447382,6.574281
6,pca90_sample10000,spectral_10,10,0,-0.070962,2.065640,88.984176,6.920037


### 1.4.5 DBSCAN

In [52]:
len(player_ids)

44905

In [53]:
pca_dbscan_results = []

eps_values = [2.0, 2.5, 3.0, 3.5, 4.0]
min_samples_values = [5, 10, 20]

for eps in eps_values:
    for min_samples in min_samples_values:

        labels, res = run_algorithm(
            X_pca90,
            "pca90",
            f"dbscan_eps{eps}_min{min_samples}",
            DBSCAN(
                eps=eps,
                min_samples=min_samples,
                n_jobs=-1
            )
        )

        res["param"] = (
            f"eps={eps}, min_samples={min_samples}"
        )

        pca_dbscan_results.append(res)

pca_dbscan_metrics = pd.concat(
    pca_dbscan_results,
    ignore_index=True
)

pca_dbscan_metrics = pca_dbscan_metrics.sort_values(
    by=["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

pca_dbscan_metrics

Running dbscan_eps2.0_min5 on pca90...
Finished dbscan_eps2.0_min5 on pca90 in 2.08s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,dbscan_eps2.0_min5,703,34268,0.472262,0.717615,436.820414,2.082122


Running dbscan_eps2.0_min10 on pca90...
Finished dbscan_eps2.0_min10 on pca90 in 2.78s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,dbscan_eps2.0_min10,279,38191,0.541907,0.647209,729.434312,2.775325


Running dbscan_eps2.0_min20 on pca90...
Finished dbscan_eps2.0_min20 on pca90 in 2.78s


...

Running dbscan_eps4.0_min10 on pca90...
Finished dbscan_eps4.0_min10 on pca90 in 2.72s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,dbscan_eps4.0_min10,281,22316,0.177833,1.419013,285.280425,2.724994


Running dbscan_eps4.0_min20 on pca90...
Finished dbscan_eps4.0_min20 on pca90 in 3.06s


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds
0,pca90,dbscan_eps4.0_min20,192,27345,0.25621,1.28818,386.710234,3.060437


,dataset,algorithm,n_clusters,n_noise,silhouette,davies_bouldin,calinski_harabasz,time_seconds,param
2,pca90,dbscan_eps2.0_min20,78,41858,0.640141,0.518741,1258.312732,2.778802,"eps=2.0, min_samples=20"
5,pca90,dbscan_eps2.5_min20,127,39678,0.546985,0.666250,874.226244,3.049625,"eps=2.5, min_samples=20"
1,pca90,dbscan_eps2.0_min10,279,38191,0.541907,0.647209,729.434312,2.775325,"eps=2.0, min_samples=10"
0,pca90,dbscan_eps2.0_min5,703,34268,0.472262,0.717615,436.820414,2.082122,"eps=2.0, min_samples=5"
4,pca90,dbscan_eps2.5_min10,372,34703,0.457313,0.807292,517.943007,2.730797,"eps=2.5, min_samples=10"
8,pca90,dbscan_eps3.0_min20,180,36634,0.453751,0.846525,608.131230,2.667251,"eps=3.0, min_samples=20"
3,pca90,dbscan_eps2.5_min5,815,30249,0.382051,0.864503,312.272808,2.718156,"eps=2.5, min_samples=5"
7,pca90,dbscan_eps3.0_min10,431,31218,0.376143,0.951280,382.282733,2.668592,"eps=3.0, min_samples=10"
11,pca90,dbscan_eps3.5_min20,202,32511,0.328560,1.076001,415.034171,2.699137,"eps=3.5, min_samples=20"
6,pca90,dbscan_eps3.0_min5,878,26442,0.304162,1.015101,235.897109,2.743950,"eps=3.0, min_samples=5"


# 2. Skup bez drzava i njegove redukcije